# Research Agent with Memory, Tools and JSON Output

This notebook builds an AI agent that:
- **remembers** the conversation (memory)
- uses **3 tools**: Wikipedia search, a blog summarizer and a calculator
- **chains prompts**: one prompt reasons, another summarizes, another formats JSON
- returns every answer as **JSON** (structured output)

## Step 0: Install
The course suggests GPT-4; I used **Google Gemini** (free API key). Switching models is one line: `ChatOpenAI(model="gpt-4")`.

In [1]:
%pip install -q "langchain-classic>=1.0" "langchain-google-genai>=3.0"

Note: you may need to restart the kernel to use updated packages.


## Step 1: Imports and the LLM
The **LLM** is the agent's brain. `langchain_classic` holds the classic agent API used in the course.

In [2]:
# Step 1: Imports and the LLM
import ast
import json
import math
import os
import re
import urllib.parse
import urllib.request
import warnings
from getpass import getpass

from langchain_classic.agents import AgentType, Tool, initialize_agent
from langchain_classic.agents.conversational_chat.output_parser import ConvoOutputParser
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

warnings.filterwarnings("ignore", category=DeprecationWarning)  # hide deprecation notices to keep the output clean
warnings.filterwarnings("ignore", message=".*sampling parameter.*")  # flash-lite ignores temperature: hide that warning

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google AI Studio API key: ")

# Free tier = 5 requests/minute, and one agent turn makes several calls.
# The rate limiter spaces the calls out so we never hit that limit.
rate_limiter = InMemoryRateLimiter(requests_per_second=4 / 60, check_every_n_seconds=0.5, max_bucket_size=1)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0, max_retries=3, rate_limiter=rate_limiter)

## Step 2: Tools
A **tool** = a Python function + a **description**. The agent reads the description to pick the right tool.

| Tool | What it does |
|---|---|
| Wikipedia Search | Real Wikipedia API: finds the page and returns its summary |
| Blog Summarizer | Finds a post in a mock blog and summarizes it with a second prompt |
| Calculator | Safe math: only numbers and operators |

In [3]:
# Step 2: The tools (Python functions the agent can call)

# --- Search tool: real Wikipedia API (find the page, then get its summary) ---
WIKI_HEADERS = {"User-Agent": "langchain-student-assignment/1.0 (educational project)"}


def _get_json(url: str) -> dict:
    request = urllib.request.Request(url, headers=WIKI_HEADERS)
    with urllib.request.urlopen(request, timeout=15) as response:
        return json.load(response)


def fetch_wikipedia_summary(topic: str) -> str:
    """
    Given a topic, return a brief summary using Wikipedia.
    Step 1: search for the best-matching page title. Step 2: fetch that page's summary.
    """
    topic = topic.strip().strip("\"'")
    try:
        query = urllib.parse.urlencode(
            {"action": "query", "list": "search", "srsearch": topic, "format": "json", "srlimit": 1})
        results = _get_json(f"https://en.wikipedia.org/w/api.php?{query}")["query"]["search"]
        if not results:
            return f"No Wikipedia page found for '{topic}'."
        title = results[0]["title"]
        page = _get_json("https://en.wikipedia.org/api/rest_v1/page/summary/"
                         + urllib.parse.quote(title.replace(" ", "_")))
        return f"Wikipedia, '{title}': {page.get('extract', '')[:1200]}"
    except Exception as exc:  # the error goes back to the agent as an Observation
        return f"Wikipedia is not reachable right now ({exc})."


def get_search_tool() -> Tool:
    return Tool(
        name="Wikipedia Search",
        func=fetch_wikipedia_summary,
        description=(
            "Use for research questions about real-world facts: places, people, history, science, "
            "geography. Input: a short topic, e.g. 'Himalayas' or 'Marie Curie'."
        ),
    )


# --- Summary tool: a mock blog + a second prompt that summarizes (prompt chaining) ---
BLOG_POSTS = {
    "remote-work": {
        "title": "Five Lessons from Three Years of Remote Work",
        "url": "https://example-blog.com/remote-work",
        "text": (
            "When our team went fully remote three years ago, we expected the hardest part to be technology. "
            "It wasn't. The real challenge was communication. Lesson one: write things down. Decisions made in "
            "a video call disappear unless someone records them, so we keep a shared decision log. Lesson two: "
            "default to asynchronous work. Not every question needs a meeting; a clear written message often "
            "saves an hour for five people. Lesson three: protect focus time. We block two meeting-free mornings "
            "per week, and our delivery speed went up by roughly 20 percent. Lesson four: be deliberate about "
            "social connection. Remote teams lose the hallway chat, so we hold a 30-minute informal call every "
            "Friday with no work topics allowed. Lesson five: measure outcomes, not hours online. People stopped "
            "worrying about looking busy once we agreed on clear weekly goals. Remote work is not easier than "
            "office work, but with these habits our team is happier and more productive than before."
        ),
    },
    "ai-healthcare": {
        "title": "How AI Is Quietly Changing Healthcare",
        "url": "https://example-blog.com/ai-healthcare",
        "text": (
            "Artificial intelligence in healthcare rarely makes headlines the way chatbots do, but its impact is "
            "growing. Hospitals use image-recognition models to flag possible problems in X-rays and scans, "
            "helping radiologists prioritise urgent cases. Other systems predict which patients are at risk of "
            "readmission so nurses can follow up early. AI also reduces paperwork: speech-to-text tools draft "
            "clinical notes, giving doctors more time with patients. The challenges are real, though. Models "
            "trained on data from one population can perform worse on another, privacy rules limit data "
            "sharing, and doctors need to understand why a model makes a recommendation before trusting it. "
            "The most successful projects treat AI as an assistant to clinicians, not a replacement."
        ),
    },
}

summary_prompt = PromptTemplate.from_template(
    "You are an editor. Summarize the blog post below in 3 bullet points followed by a one-sentence "
    "takeaway. Keep it under 90 words and do not add facts that are not in the post.\n\n"
    "Title: {title}\n\nPost:\n{text}"
)
summary_chain = summary_prompt | llm | StrOutputParser()   # chain: prompt -> model -> plain text


def summarize_blog(query: str) -> str:
    """Find the blog post that best matches the query (title, URL or keywords) and summarize it."""
    words = set(re.findall(r"[a-z]+", query.lower()))
    best_post, best_score = None, 0
    for slug, post in BLOG_POSTS.items():
        keywords = set(re.findall(r"[a-z]+", f"{slug} {post['title']} {post['url']}".lower()))
        score = len(words & keywords)
        if score > best_score:
            best_post, best_score = post, score
    if best_post is None:
        titles = "; ".join(p["title"] for p in BLOG_POSTS.values())
        return f"No matching blog post. Available posts: {titles}"
    summary = summary_chain.invoke({"title": best_post["title"], "text": best_post["text"]})
    return f"Summary of '{best_post['title']}' ({best_post['url']}):\n{summary}"


def get_summary_tool() -> Tool:
    return Tool(
        name="Blog Summarizer",
        func=summarize_blog,
        description=(
            "Use when the user asks to summarize a blog post or article. Input: the post's topic, "
            "title or URL, e.g. 'remote work'."
        ),
    )


# --- Calculator: only numbers and math operators get to eval() ---
ALLOWED_NODES = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.Add, ast.Sub, ast.Mult,
                 ast.Div, ast.FloorDiv, ast.Mod, ast.Pow, ast.USub, ast.UAdd)


def calculator(expression: str) -> str:
    """Evaluate a plain arithmetic expression safely (no names, no function calls)."""
    expr = re.sub(r"(?<=\d),(?=\d{3})", "", expression.strip().strip("\"'")).replace("^", "**")
    try:
        tree = ast.parse(expr, mode="eval")
        if not all(isinstance(node, ALLOWED_NODES) for node in ast.walk(tree)):
            return "Error: only numbers and + - * / ** % are allowed."
        result = eval(compile(tree, "<calculator>", "eval"), {"__builtins__": {}}, {})
        return f"{round(result, 4):,}"
    except Exception as exc:
        return f"Error: could not evaluate '{expression}' ({exc})."

In [4]:
# Wrap each function as a LangChain Tool (name + description + function)
calculator_tool = Tool(
    name="Calculator",
    func=calculator,
    description="Use for any arithmetic. Input: a plain math expression with numbers only, e.g. '8849 * 3.28084'.",
)

tools = [get_search_tool(), get_summary_tool(), calculator_tool]

# Test the tools alone (no agent yet)
print("Calculator('8,849 * 3.28084') ->", calculator("8,849 * 3.28084"))
print("Calculator(\"__import__('os')\") ->", calculator("__import__('os')"))
print("Wikipedia('Himalayas')        ->", fetch_wikipedia_summary("Himalayas")[:160], "...")

Calculator('8,849 * 3.28084') -> 29,032.1532
Calculator("__import__('os')") -> Error: only numbers and + - * / ** % are allowed.


Wikipedia('Himalayas')        -> Wikipedia, 'Himalayas': The Himalayas, or Himalaya, is a mountain range in Asia separating the plains of the Indian subcontinent from the Tibetan Plateau. The r ...


## Step 3: Memory and prompts
- **Memory** stores every message, so "the range we discussed" makes sense later.
- **System prompt** = the agent's rules. **User prompt** = the user's question.
- I use the *chat* agent type because this memory stores real chat messages.

In [5]:
# Step 3: Memory (keeps the chat as messages) + system prompt (the agent's rules)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

SYSTEM_PROMPT = """You are a helpful research assistant.

How to choose a tool:
- Factual or research questions (places, people, history, science) -> use "Wikipedia Search". Do not answer from memory.
- Requests to summarize a blog post or article -> use "Blog Summarizer".
- Any arithmetic -> use "Calculator". Never do math in your head.
- Questions about the conversation itself ("what have we discussed?") -> answer from the chat history, no tool.

If the user refers to something said earlier ("it", "that mountain range"), use the chat history to work out what they mean.
Call one tool at a time and wait for its result before answering. Keep final answers short and factual."""

## Step 4: The agent
`initialize_agent` returns an **AgentExecutor**, which runs the loop *think → use a tool → read the result → answer*.

**Fix:** Gemini sometimes invented a tool's result. `FirstActionParser` runs only the first action, so answers come from real tool results.

In [6]:
# Step 4: Create the agent = LLM + tools + memory + system prompt

class FirstActionParser(ConvoOutputParser):
    """Read only the FIRST {"action": ..., "action_input": ...} blob and ignore anything the model wrote after it."""

    def parse(self, text):
        start = text.find("{")
        if start != -1:
            try:
                blob, end = json.JSONDecoder().raw_decode(text[start:])
            except ValueError:
                blob = None
            if isinstance(blob, dict) and {"action", "action_input"} <= blob.keys():
                log = text[: start + end]
                if blob["action"] == "Final Answer":
                    return AgentFinish({"output": str(blob["action_input"])}, log)
                return AgentAction(blob["action"], blob["action_input"], log)
        return super().parse(text)


agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    agent_kwargs={"system_message": SYSTEM_PROMPT, "output_parser": FirstActionParser()},
    verbose=True,                 # show every thought, tool call and tool result
    handle_parsing_errors=True,   # if the model breaks the format, retry instead of crashing
    max_iterations=6,             # stop after 6 steps (avoids infinite loops)
)

# Print the prompt structure: system -> history -> user -> scratchpad
for message in agent_executor.agent.llm_chain.prompt.messages:
    print("-", type(message).__name__)

- SystemMessagePromptTemplate
- MessagesPlaceholder
- HumanMessagePromptTemplate
- MessagesPlaceholder


## Step 5: Structured output
A second prompt turns the free-text answer into **JSON** with fixed fields. `tools_used` comes from a callback that records the tools that really ran.

In [7]:
response_schemas = [
    ResponseSchema(name="answer", description="the answer to the user's question in one or two sentences"),
    ResponseSchema(name="topic", description="the main topic of the question in 1-3 words"),
    ResponseSchema(name="follow_up_question", description="one useful follow-up question the user could ask next"),
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

structure_prompt = PromptTemplate(
    template=(
        "Turn the assistant's answer into structured data. Do not add new facts.\n"
        "{format_instructions}\n\nUser question: {question}\nAssistant answer: {answer}"
    ),
    input_variables=["question", "answer"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)
structure_chain = structure_prompt | llm | output_parser   # chain: prompt -> model -> parser (returns a dict)


class ToolTracker(BaseCallbackHandler):
    """Records the name of every tool the agent runs."""

    def __init__(self):
        self.tools_used = []

    def on_tool_start(self, serialized, input_str, **kwargs):
        self.tools_used.append((serialized or {}).get("name") or kwargs.get("name"))


print(output_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"answer": string  // the answer to the user's question in one or two sentences
	"topic": string  // the main topic of the question in 1-3 words
	"follow_up_question": string  // one useful follow-up question the user could ask next
}
```


## Step 6: Chat loop
- `ask()` runs one turn with `agent_executor.run()` and prints the JSON.
- `chat_loop()` repeats until you type `exit`.

In [8]:
def ask(question: str) -> dict:
    """One turn: run the agent, then convert its answer into structured JSON."""
    print(f"\n🧑 USER: {question}")
    tracker = ToolTracker()
    answer = agent_executor.run(question, callbacks=[tracker])
    result = structure_chain.invoke({"question": question, "answer": answer})
    result["tools_used"] = tracker.tools_used
    print("\n🤖 STRUCTURED OUTPUT:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result


def chat_loop(scripted_inputs=None):
    """Interactive loop. Type 'exit' to stop. Pass a list of messages to simulate typing."""
    inputs = iter(scripted_inputs) if scripted_inputs is not None else None
    while True:
        question = next(inputs, "exit") if inputs else input("You (type 'exit' to stop): ")
        if question.strip().lower() in {"exit", "quit"}:
            print("👋 Goodbye!")
            break
        ask(question)

## Step 7: Test the agent

| Test | What it checks |
|---|---|
| 1. *Tell me about the Himalayas* | Research question → **Wikipedia Search** |
| 2. *Summarize the blog post about remote work* | Blog summary → **Blog Summarizer** (a chain inside a tool) |
| 3. *The highest peak in the mountain range we discussed…* | **Memory** ("the range we discussed" = the Himalayas) + **Calculator** |
| 4. *What topics have we discussed so far?* | **Memory** only, run through the interactive `chat_loop()` |

In [9]:
# Test 1: research question -> Wikipedia Search
ask("Tell me about the Himalayas");


🧑 USER: Tell me about the Himalayas


> Entering new AgentExecutor chain...


```json
{
    "action": "Wikipedia Search",
    "action_input": "Himalayas"
}


Observation: Wikipedia, 'Himalayas': The Himalayas, or Himalaya, is a mountain range in Asia separating the plains of the Indian subcontinent from the Tibetan Plateau. The range has some of the highest peaks on Earth, including the highest, Mount Everest. More than 100 peaks exceeding elevations of 7,200 metres above sea level lie in the Himalayas.
Thought:

```json
{
    "action": "Final Answer",
    "action_input": "According to Wikipedia, the Himalayas are a mountain range in Asia that separate the plains of the Indian subcontinent from the Tibetan Plateau. The range includes some of the highest peaks on Earth, such as Mount Everest, with more than 100 peaks exceeding elevations of 7,200 metres above sea level."
}

> Finished chain.



🤖 STRUCTURED OUTPUT:
{
  "answer": "The Himalayas are an Asian mountain range separating the Indian subcontinent from the Tibetan Plateau. They contain Earth's highest peaks, including Mount Everest, with over 100 peaks exceeding 7,200 metres in elevation.",
  "topic": "The Himalayas",
  "follow_up_question": "How were the Himalayas formed?",
  "tools_used": [
    "Wikipedia Search"
  ]
}


In [10]:
# Test 2: summarize a blog -> Blog Summarizer
ask("Summarize the blog post about remote work");


🧑 USER: Summarize the blog post about remote work


> Entering new AgentExecutor chain...


```json
{
    "action": "Blog Summarizer",
    "action_input": "remote work"
}


Observation: Summary of 'Five Lessons from Three Years of Remote Work' (https://example-blog.com/remote-work):
* Prioritize written communication, shared decision logs, and asynchronous messaging over unnecessary meetings.
* Boost productivity and engagement by protecting focus time and scheduling intentional, non-work social calls.
* Measure success through clear weekly outcomes rather than monitoring hours spent online.

**Takeaway:** Adopting intentional habits around communication, focus, and outcome-based goals made the remote team happier and more productive.
Thought:

```json
{
    "action": "Final Answer",
    "action_input": "Based on the blog post 'Five Lessons from Three Years of Remote Work', the key takeaways are:\n* Prioritize written communication, shared decision logs, and asynchronous messaging instead of unnecessary meetings.\n* Boost productivity and engagement by protecting focus time and scheduling intentional, non-work social calls.\n* Measure success through clear weekly outcomes rather than monitoring hours spent online.\n\nOverall, adopting intentional habits around communication, focus, and outcome-based goals made the remote team happier and more productive."
}

> Finished chain.



🤖 STRUCTURED OUTPUT:
{
  "answer": "The blog post highlights that adopting intentional habits around written communication, protected focus time, and outcome-based goals leads to a happier and more productive remote team.",
  "topic": "Remote work",
  "follow_up_question": "What are some specific strategies for improving asynchronous communication on a remote team?",
  "tools_used": [
    "Blog Summarizer"
  ]
}


In [11]:
# Test 3: memory + tool -> "the range we discussed" comes from memory, the math from the Calculator
ask("The highest peak in the mountain range we discussed is 8,849 meters tall. How many feet is that? (1 m = 3.28084 ft)");


🧑 USER: The highest peak in the mountain range we discussed is 8,849 meters tall. How many feet is that? (1 m = 3.28084 ft)


> Entering new AgentExecutor chain...


```json
{
    "action": "Calculator",
    "action_input": "8849 * 3.28084"
}
Observation: 29,032.1532
Thought:

```json
{
    "action": "Final Answer",
    "action_input": "Mount Everest, the highest peak in the Himalayas, stands at 8,849 meters, which is approximately 29,032.15 feet tall."
}

> Finished chain.



🤖 STRUCTURED OUTPUT:
{
  "answer": "Mount Everest, the highest peak in the Himalayas at 8,849 meters, is approximately 29,032.15 feet tall.",
  "topic": "Mount Everest height",
  "follow_up_question": "What is the prominence of Mount Everest?",
  "tools_used": [
    "Calculator"
  ]
}


In [12]:
# Test 4: the chat loop with scripted input (instead of typing) -> memory only, no tool
chat_loop(["What topics have we discussed so far?", "exit"])

# To chat live, run: chat_loop()


🧑 USER: What topics have we discussed so far?


> Entering new AgentExecutor chain...


```json
{
    "action": "Final Answer",
    "action_input": "So far, we have discussed the Himalayas, summarized a blog post about remote work, and converted the height of Mount Everest from meters to feet."
}

> Finished chain.



🤖 STRUCTURED OUTPUT:
{
  "answer": "So far, our discussions have covered the Himalayas, a blog post about remote work, and the conversion of Mount Everest's height from meters to feet.",
  "topic": "Discussion History",
  "follow_up_question": "Would you like to dive deeper into any of these previous topics?",
  "tools_used": []
}
👋 Goodbye!


In [13]:
# The memory: these messages are added to the prompt on every turn
for message in memory.chat_memory.messages:
    print(f"{type(message).__name__:>12}: {message.content[:110]}")

HumanMessage: Tell me about the Himalayas
   AIMessage: According to Wikipedia, the Himalayas are a mountain range in Asia that separate the plains of the Indian subc
HumanMessage: Summarize the blog post about remote work
   AIMessage: Based on the blog post 'Five Lessons from Three Years of Remote Work', the key takeaways are:
* Prioritize wri
HumanMessage: The highest peak in the mountain range we discussed is 8,849 meters tall. How many feet is that? (1 m = 3.2808
   AIMessage: Mount Everest, the highest peak in the Himalayas, stands at 8,849 meters, which is approximately 29,032.15 fee
HumanMessage: What topics have we discussed so far?
   AIMessage: So far, we have discussed the Himalayas, summarized a blog post about remote work, and converted the height of


## What I learned
- **Check the trace, not just the answer:** the model claimed to use tools it never ran.
- **The memory format must match the agent type** (chat messages → chat agent).
- **Free APIs have limits:** 5 requests/minute, so I added a rate limiter.
- **Small prompts chained together** are simpler and more reliable than one big prompt.
- **JSON output** makes the agent usable by other apps (e.g. a support bot that creates tickets).